# ML Experiment Benchmark

Small rendered ML benchmark using the project preprocessing and classifier helpers.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from gtd_capstone.data.repository import DataRepository
from gtd_capstone.features.store import materialize_feature_set
from gtd_capstone.ml.experiment_suite import FEATURES, classifier, preprocess

incidents = DataRepository().load_incidents(sample_rows=min(SAMPLE_ROWS, 4000))
frame = materialize_feature_set(incidents)
x = frame[FEATURES]
y = frame["severity"]
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y if y.value_counts().min() >= 2 else None,
)
rows = []
for family in ["logistic_regression", "random_forest"]:
    model = Pipeline([("preprocess", preprocess()), ("model", classifier(family))])
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    rows.append(
        {
            "family": family,
            "accuracy": accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        }
    )
pd.DataFrame(rows)

,family,accuracy,macro_f1,weighted_f1
0,logistic_regression,0.687,0.368832,0.761999
1,random_forest,0.840,0.454929,0.850258


In [3]:
metrics_path = ROOT / "artifacts" / "models" / "severity_metrics.json"
if metrics_path.exists():
    json.loads(metrics_path.read_text())["metrics"]
else:
    {"note": "No persisted severity metrics artifact found."}